In [1]:
import sys, pandas, duckdb, pyarrow
print(sys.executable)
print(pandas.__version__, duckdb.__version__, pyarrow.__version__)

/Users/rajkandula/projects/walmart-demand-forecasting/.venv/bin/python
3.0.6 1.5.5 25.0.1


In [2]:
import sys; sys.path.append("..")
import duckdb
from src.config import RAW_DIR, PROCESSED_DIR

con = duckdb.connect()

In [3]:
con.sql(f"""
CREATE OR REPLACE VIEW sales AS
SELECT * FROM read_parquet('{PROCESSED_DIR / "tx_sales_long.parquet"}')
""")

con.sql(f"""
CREATE OR REPLACE VIEW calendar AS
SELECT * FROM read_csv('{RAW_DIR / "calendar.csv"}')
""")

con.sql(f"""
CREATE OR REPLACE VIEW prices AS
SELECT * FROM read_csv('{RAW_DIR / "sell_prices.csv"}')
WHERE store_id IN ('TX_1', 'TX_2', 'TX_3')
""")

In [4]:
cal_rows, cal_keys = con.sql(
    "SELECT COUNT(*), COUNT(DISTINCT d) FROM calendar"
).fetchone()
assert cal_rows == cal_keys

dup_price_keys = con.sql("""
SELECT COUNT(*) FROM (
    SELECT store_id, item_id, wm_yr_wk
    FROM prices
    GROUP BY store_id, item_id, wm_yr_wk
    HAVING COUNT(*) > 1
)
""").fetchone()[0]
assert dup_price_keys == 0
print(cal_rows, dup_price_keys)

1969 0


In [5]:
con.sql("""
CREATE OR REPLACE VIEW tx_daily AS
SELECT
    s.*,
    c.date, c.wm_yr_wk, c.weekday, c.wday, c.month, c.year,
    c.event_name_1, c.event_type_1, c.event_name_2, c.event_type_2,
    c.snap_TX,
    p.sell_price
FROM sales AS s
LEFT JOIN calendar AS c
    ON s.d::VARCHAR = c.d
LEFT JOIN prices AS p
    ON s.store_id::VARCHAR = p.store_id
   AND s.item_id::VARCHAR = p.item_id
   AND c.wm_yr_wk = p.wm_yr_wk
""")

In [6]:
checks = con.sql("""
SELECT
    COUNT(*),
    SUM(sales),
    COUNT(*) FILTER (WHERE date IS NULL),
    COUNT(*) FILTER (WHERE sell_price IS NULL),
    COUNT(*) FILTER (WHERE sell_price IS NULL AND sales > 0)
FROM tx_daily
""").fetchone()
n_rows, total_units, missing_date, missing_price, sold_without_price = checks
print(checks)

assert n_rows == 17_754_327
assert total_units == 19_228_405
assert missing_date == 0

(17754327, 19228405, 0, 3411324, 0)


In [7]:
daily_path = PROCESSED_DIR / "tx_daily.parquet"
con.sql(f"""
COPY (SELECT * FROM tx_daily ORDER BY id, date)
TO '{daily_path}' (FORMAT parquet)
""")
print(con.sql(f"SELECT COUNT(*) FROM read_parquet('{daily_path}')").fetchone()[0],
      round(daily_path.stat().st_size / 1e6, 2), "MB")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

17754327 62.44 MB


In [8]:
con.sql("""
CREATE OR REPLACE VIEW tx_weekly AS
SELECT
    id, item_id, dept_id, cat_id, store_id, wm_yr_wk,
    MIN(date) AS week_start,
    COUNT(*) AS n_days,
    SUM(sales) AS sales,
    AVG(sell_price) AS avg_price,
    COUNT(sell_price) AS days_priced,
    SUM(snap_TX) AS snap_days,
    COUNT(event_name_1) AS event_days
FROM tx_daily
GROUP BY ALL
""")

con.sql("SELECT n_days, COUNT(*) AS n_rows FROM tx_weekly GROUP BY n_days ORDER BY n_days").show()

weekly_path = PROCESSED_DIR / "tx_weekly.parquet"
con.sql(f"""
COPY (SELECT * FROM tx_weekly WHERE n_days = 7 ORDER BY id, wm_yr_wk)
TO '{weekly_path}' (FORMAT parquet)
""")
print(con.sql(f"""
SELECT COUNT(*), COUNT(DISTINCT id), COUNT(DISTINCT wm_yr_wk), SUM(sales)
FROM read_parquet('{weekly_path}')
""").fetchone())

┌────────┬─────────┐
│ n_days │ n_rows  │
│ int64  │  int64  │
├────────┼─────────┤
│      2 │    9147 │
│      7 │ 2533719 │
└────────┴─────────┘



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(2533719, 9147, 277, 19199909.0)


In [9]:
con.sql("""
CREATE OR REPLACE VIEW tx_weekly AS
SELECT
    id, item_id, dept_id, cat_id, store_id, wm_yr_wk,
    MIN(date) AS week_start,
    COUNT(*) AS n_days,
    SUM(sales)::INTEGER AS sales,
    AVG(sell_price) AS avg_price,
    COUNT(sell_price) AS days_priced,
    SUM(snap_TX)::INTEGER AS snap_days,
    COUNT(event_name_1) AS event_days
FROM tx_daily
GROUP BY ALL
""")

con.sql("SELECT n_days, COUNT(*) AS n_rows FROM tx_weekly GROUP BY n_days ORDER BY n_days").show()

weekly_path = PROCESSED_DIR / "tx_weekly.parquet"
con.sql(f"""
COPY (SELECT * FROM tx_weekly WHERE n_days = 7 ORDER BY id, wm_yr_wk)
TO '{weekly_path}' (FORMAT parquet)
""")
print(con.sql(f"""
SELECT COUNT(*), COUNT(DISTINCT id), COUNT(DISTINCT wm_yr_wk), SUM(sales)
FROM read_parquet('{weekly_path}')
""").fetchone())

┌────────┬─────────┐
│ n_days │ n_rows  │
│ int64  │  int64  │
├────────┼─────────┤
│      2 │    9147 │
│      7 │ 2533719 │
└────────┴─────────┘



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(2533719, 9147, 277, 19199909)
